# SweepMeasurementBuilder Example

This is an example of building a `PulseSchedule` from a `SweepMeasurementConfig` using `SweepMeasurementBuilder`.

In [ ]:
import numpy as np
import tunits

from qubex.measurement import SweepMeasurementBuilder
from qubex.measurement.models.sweep_measurement_config import (
    DataAcquisitionConfig,
    FrequencyConfig,
    ParameterSweepConfig,
    ParameterSweepContent,
    ParametricSequenceConfig,
    ParametricSequencePulseCommand,
    SweepMeasurementConfig,
)

In [ ]:
# Define a parametric sequence

sequence = ParametricSequenceConfig(
    delta_time=tunits.Time(2.0, "ns"),
    variable_list=["amplitude", "duration"],
    command_list=[
        ParametricSequencePulseCommand(
            name="Gaussian",
            channel_list=["Q01"],
            argument_list=[32, 1.0, 4.0],
        ),
        ParametricSequencePulseCommand(
            name="Blank",
            channel_list=["Q01"],
            argument_list=["duration"],
        ),
        ParametricSequencePulseCommand(
            name="Gaussian",
            channel_list=["Q01"],
            argument_list=[32, "amplitude", 4.0],
        ),
        ParametricSequencePulseCommand(
            name="Barrier",
            channel_list=[],
            argument_list=[],
        ),
        ParametricSequencePulseCommand(
            name="FlatTop",
            channel_list=["Q02"],
            argument_list=[128, "amplitude", 16.0],
        ),
    ],
)
sequence

In [ ]:
# Define sweep settings (sweep amp)
sweep = ParameterSweepConfig(
    sweep_content_list={
        "amplitude_sweep": ParameterSweepContent(
            category="sequence_variable",
            sweep_target=["amplitude"],
            value_list=np.linspace(0.5, 2.0, 4),
        ),
        "duration_sweep": ParameterSweepContent(
            category="sequence_variable",
            sweep_target=["duration"],
            value_list=np.arange(32, 128, 32),
        ),
        "frequency_sweep": ParameterSweepContent(
            category="frequency_shift",
            sweep_target=["Q01"],
            value_list=np.linspace(-0.05, 0.05, 4),
        ),
    },
    sweep_axis=[["amplitude_sweep", "frequency_sweep"], ["duration_sweep"]],
)
sweep

In [ ]:
frequency = FrequencyConfig(
    channel_to_frequency={"Q01": tunits.Frequency(5.0, "GHz")},
    channel_to_frequency_reference={},
    channel_to_frequency_shift={"Q01": tunits.Frequency(10.0, "MHz")},
    keep_oscillator_relative_phase=True,
)

data_acq = DataAcquisitionConfig(
    shot_count=1000,
    shot_repetition_margin=tunits.Time(300, "us"),
    data_acquisition_duration=tunits.Time(512, "ns"),
    data_acquisition_delay=tunits.Time(128, "ns"),
    data_acquisition_timeout=tunits.Time(10, "s"),
    flag_average_waveform=True,
    flag_average_shots=True,
    delta_time=tunits.Time(2.0, "ns"),
    channel_to_averaging_time={"Q01": tunits.Time(236, "ns")},
    channel_to_averaging_window={"Q01": np.array([0.0 + 0.0j, 1.0 + 1.0j])},
)

config = SweepMeasurementConfig(
    channel_list=["Q01", "Q02"],
    sequence=sequence,
    frequency=frequency,
    data_acquisition=data_acq,
    sweep_parameter=sweep,
)
config

In [ ]:
config.to_dict()

In [ ]:
config.to_json()

In [ ]:
builder = SweepMeasurementBuilder(config=config)
print(builder.sweep_shape)

schedule = builder.build_schedule(indices=(3, 2))
schedule.plot()

In [ ]:
builder.resolve_sweep_state(sweep_indices=(3, 2))

In [ ]:
schedule.get_sequences()

In [ ]:
schedule.get_sampled_sequences()

In [ ]:
for indices in builder.iterate():
    schedule = builder.build_schedule(indices=indices)
    print(f"Sweep indices: {indices}")
    schedule.plot()